# 01 · Data overview
What's in the SkillCorner A-League 2024/25 open data: matches, players, dynamic events,
phases of play and season aggregates. Tracking gets its own notebook (02).

In [ ]:
import sys
from pathlib import Path

# Make the repo's src/ importable no matter where the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src.config import find_data_dir
from src.loading import (match_table, list_matches, load_match_meta, load_events, load_phases,
                         load_all, list_aggregates, load_aggregate)

print("Data folder:", find_data_dir())
matches = match_table()
matches

## Files per match
Tracking files are large; dynamic events and phases are small CSVs.

In [ ]:
rows = []
for mid in list_matches():
    folder = find_data_dir() / "matches" / str(mid)
    for f in folder.iterdir():
        kind = f.name.split("_", 1)[1].rsplit(".", 1)[0]
        rows.append({"match_id": mid, "file": kind, "MB": f.stat().st_size / 1e6})
files = pd.DataFrame(rows).pivot_table(index="match_id", columns="file", values="MB").round(1)
files

## Players
Roles, positions and minutes from each match's metadata.

In [ ]:
players = pd.concat([load_match_meta(m)["players"].assign(match_id=m) for m in list_matches()],
                    ignore_index=True)
print(f"{players.player_id.nunique()} unique players across {players.match_id.nunique()} matches")
players.groupby("position_group", dropna=False).agg(
    appearances=("player_id", "size"), players=("player_id", "nunique")).sort_values("appearances", ascending=False)

## Dynamic events
Four event types. `on_ball_engagement` is the defending one: pressing, pressure,
counter-pressing and recovery pressing on the player in possession.

In [ ]:
events = load_all(load_events)
print(f"{len(events):,} events, {events.shape[1]} columns")
events.groupby(["event_type", "event_subtype"], dropna=False).size().rename("n").to_frame()

In [ ]:
# Events per match by type
events.pivot_table(index="match_id", columns="event_type", values="event_id", aggfunc="size")

In [ ]:
# Which columns are populated for which event type? (share non-missing)
filled = events.groupby("event_type").apply(lambda d: d.notna().mean(), include_groups=False).T
filled[(filled > 0.5).any(axis=1)].round(2)

## Phases of play
Each phase has an in-possession type (e.g. build_up, create) and an out-of-possession type
(e.g. high_block, medium_block, low_block), useful for splitting defending by context.

In [ ]:
phases = load_all(load_phases)
print(f"{len(phases):,} phases")
(phases.groupby("team_out_of_possession_phase_type")
       .agg(phases=("index", "size"), minutes=("duration", lambda s: s.sum() / 60),
            median_seconds=("duration", "median"),
            share_lead_to_shot=("team_possession_lead_to_shot", "mean"))
       .sort_values("minutes", ascending=False).round(2))

## Season aggregates
Player-season CSVs (physical, off-ball runs, passing), performances over 60 minutes.

In [ ]:
aggs = list_aggregates()
if not aggs:
    print("No aggregates folder found.")
for path in aggs:
    df = load_aggregate(path)
    print(f"{path.name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head(3))